# Leveraged ETF Pairs - Options Arbitrage Analysis

**Goal:** Identify top opportunities for IV arbitrage across multiple leveraged/inverse ETF pairs and calculate PnL projections.

In [1]:
# 1. Setup and Imports
import os
import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import timedelta
from typing import List, Dict, Optional, Union
from IPython.display import display, HTML
from time import sleep
import warnings
import importlib
warnings.filterwarnings('ignore')

# Import helper functions
import etf_helpers
import visualization_helpers

# Reload modules to get latest changes
importlib.reload(etf_helpers)
importlib.reload(visualization_helpers)

from etf_helpers import (
    standardize_pair_name,
    calculate_volatility_decay,
    calculate_pnl_with_decay,
    calculate_breakeven_price,
    find_delta_neutral_pair
)
from visualization_helpers import (
    create_3d_surface_plot,
    create_pnl_profile_plots,
    create_opportunity_summary_plot,
    export_interactive_html
)

# Configure pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 1000)

print("✓ Libraries and helper functions imported successfully")


✓ Libraries and helper functions imported successfully


In [ ]:
# 2. Configuration
TIMESTAMP = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
HTML_OUTPUT_DIR = os.path.join(os.getcwd(), 'html_reports')
CACHE_DIR = os.path.join(os.getcwd(), 'cache')
os.makedirs(HTML_OUTPUT_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

# ETF Pairs - Top liquid leveraged/inverse pairs with underlying volatility estimates
# Includes: 3x leveraged pairs, 1x/-1x inverse pairs, and individual stocks paired with sector inverse ETFs
ETF_PAIRS = [
    # 3x Leveraged/Inverse Pairs
    {"bull": "UPRO", "bear": "SPXU", "name": "S&P 500 3x", "bull_ric": "UPRO.K", "bear_ric": "SPXU.K", "underlying_vol": 0.15, "leverage": 3.0},
    {"bull": "TQQQ", "bear": "SQQQ", "name": "Nasdaq-100 3x", "bull_ric": "TQQQ.K", "bear_ric": "SQQQ.K", "underlying_vol": 0.22, "leverage": 3.0},
    {"bull": "SOXL", "bear": "SOXS", "name": "Semiconductor 3x", "bull_ric": "SOXL.K", "bear_ric": "SOXS.K", "underlying_vol": 0.30, "leverage": 3.0},
    {"bull": "TNA", "bear": "TZA", "name": "Russell 2000 3x", "bull_ric": "TNA.K", "bear_ric": "TZA.K", "underlying_vol": 0.18, "leverage": 3.0},
    {"bull": "FAS", "bear": "FAZ", "name": "Financial 3x", "bull_ric": "FAS.K", "bear_ric": "FAZ.K", "underlying_vol": 0.25, "leverage": 3.0},

    # 1x/-1x ETF/Inverse Pairs
    {"bull": "SPY", "bear": "SH", "name": "S&P 500 1x/-1x", "bull_ric": "SPY.K", "bear_ric": "SH.K", "underlying_vol": 0.15, "leverage": 1.0},
    {"bull": "QQQ", "bear": "PSQ", "name": "Nasdaq-100 1x/-1x", "bull_ric": "QQQ.O", "bear_ric": "PSQ.K", "underlying_vol": 0.22, "leverage": 1.0},
    {"bull": "IWM", "bear": "RWM", "name": "Russell 2000 1x/-1x", "bull_ric": "IWM.K", "bear_ric": "RWM.K", "underlying_vol": 0.18, "leverage": 1.0},
    {"bull": "DIA", "bear": "DOG", "name": "Dow Jones 1x/-1x", "bull_ric": "DIA.K", "bear_ric": "DOG.K", "underlying_vol": 0.14, "leverage": 1.0},
    {"bull": "XLF", "bear": "SEF", "name": "Financials 1x/-1x", "bull_ric": "XLF.K", "bear_ric": "SEF.K", "underlying_vol": 0.25, "leverage": 1.0},
    {"bull": "XLE", "bear": "ERY", "name": "Energy 1x/-2x", "bull_ric": "XLE.K", "bear_ric": "ERY.K", "underlying_vol": 0.35, "leverage": 1.0},

    # Individual Stocks paired with Sector Inverse ETFs
    {"bull": "NVDA", "bear": "SOXS", "name": "NVDA / Semiconductor -3x", "bull_ric": "NVDA.O", "bear_ric": "SOXS.K", "underlying_vol": 0.45, "leverage": 1.0},
    {"bull": "GOOGL", "bear": "SQQQ", "name": "GOOGL / Nasdaq-100 -3x", "bull_ric": "GOOGL.O", "bear_ric": "SQQQ.K", "underlying_vol": 0.35, "leverage": 1.0},
    {"bull": "AAPL", "bear": "SQQQ", "name": "AAPL / Nasdaq-100 -3x", "bull_ric": "AAPL.O", "bear_ric": "SQQQ.K", "underlying_vol": 0.30, "leverage": 1.0},
    {"bull": "TSLA", "bear": "SQQQ", "name": "TSLA / Nasdaq-100 -3x", "bull_ric": "TSLA.O", "bear_ric": "SQQQ.K", "underlying_vol": 0.60, "leverage": 1.0},
    {"bull": "AMD", "bear": "SOXS", "name": "AMD / Semiconductor -3x", "bull_ric": "AMD.O", "bear_ric": "SOXS.K", "underlying_vol": 0.50, "leverage": 1.0},
]

# Extract tickers and build option list codes
UNDERLYING_TICKERS = []
UNDERLYING_RICS = []
OPTION_LISTS = []

for pair in ETF_PAIRS:
    UNDERLYING_TICKERS.extend([pair["bull"], pair["bear"]])
    UNDERLYING_RICS.extend([pair["bull_ric"], pair["bear_ric"]])
    # Datastream option list codes: LOPT{TICKER}LC for calls, LOPT{TICKER}LP for puts
    for ticker in [pair["bull"], pair["bear"]]:
        OPTION_LISTS.extend([f'LOPT{ticker}LC', f'LOPT{ticker}LP'])

# Trading Parameters
RISK_FREE_RATE = 0.045
EROSION = 0.05  # Annual erosion rate for leveraged ETFs (baseline)
MONEYNESS_MIN = -0.15
MONEYNESS_MAX = 0.15
MAX_DTE = 360
MIN_DTE = 7
# Note: Leverage is now pair-specific (stored in ETF_PAIRS)

print(f"✓ Configuration loaded for {len(ETF_PAIRS)} ETF pairs")
print(f"  Tickers: {len(set(UNDERLYING_TICKERS))} unique tickers")
print(f"  Option Lists: {len(OPTION_LISTS)} lists")
print(f"  Output: {HTML_OUTPUT_DIR}")
print(f"  Cache: {CACHE_DIR}")


✓ Configuration loaded for 16 ETF pairs
  Tickers: 27 unique tickers
  Option Lists: 64 lists
  Output: c:\Users\P3004204\OneDrive - Corporate\Workspace\python\LSEG_Options_Analysis\html_reports
  Cache: c:\Users\P3004204\OneDrive - Corporate\Workspace\python\LSEG_Options_Analysis\cache


In [3]:
# 3. Load Credentials and Initialize Sessions
import sys
import DatastreamPy as DSpy
import lseg.data as ld

# Import implied volatility functions
from implied_volatility import implied_volatility, black_scholes_price, implied_risk_free_rate

# Load credentials from credential file
parent_dir = os.path.dirname(os.getcwd())
credential_path = os.path.join(parent_dir, 'Refinitiv')
sys.path.insert(0, credential_path)

try:
    from credential import DS_user, DS_password, LOGIN, PASSWORD, KEY
    print("✓ Credentials loaded")
except ImportError:
    print("✗ Could not import credentials")
    raise

# Initialize Datastream (for option list retrieval)
try:
    ds = DSpy.DataClient(username=DS_user, password=DS_password)
    print("✓ Datastream initialized")
except Exception as e:
    print(f"✗ Datastream error: {e}")
    raise

# Initialize LSEG Data Library (for live pricing)
CONFIG_PATH = "../LsgeDataApi/Configuration"
os.environ["LD_LIB_CONFIG_PATH"] = CONFIG_PATH

try:
    # Check if session is already open
    try:
        # Try a simple call to test if session is active
        test = ld.get_data(universe=['UPRO.K'], fields=['CF_LAST'])
        print("✓ LSEG session already open and active")
    except:
        # Session not open, open it
        ld.open_session()
        print("✓ LSEG session opened")
except Exception as e:
    print(f"✗ LSEG session error: {e}")
    raise


✓ Credentials loaded
✓ Datastream initialized
✓ LSEG session opened


In [4]:
# 4. Retrieve Options Data from Datastream and LSEG

# Helper functions (from options_analysis_lseg.ipynb)
def calculate_dte(expiry_date):
    """Calculate days to expiration."""
    if isinstance(expiry_date, str):
        expiry = datetime.datetime.strptime(expiry_date, '%Y-%m-%d')
    else:
        expiry = expiry_date
    today = datetime.datetime.now()
    dte = (expiry - today).days
    return dte

def get_live_options_quotes(rics):
    """Retrieve live quotes for options with standardized column names."""
    if not rics:
        return pd.DataFrame()

    fields = ['BID', 'ASK', 'PRIMACT_1', 'IMP_VOLT', 'STRIKE_PRC',
              'EXPIR_DATE', 'PUTCALLIND', 'ACVOL_1', 'CF_NAME']

    try:
        df = ld.get_data(universe=rics, fields=fields)
        if df.empty:
            return df

        # Standardize column names
        column_mapping = {
            'Instrument': 'RIC',
            'CF_NAME': 'NAME',
            'EXPIR_DATE': 'EXP',
            'STRIKE_PRC': 'Strike',
            'PUTCALLIND': 'CP',
            'PRIMACT_1': 'MP',
            'IMP_VOLT': 'VL',
            'ASK': 'PA',
            'BID': 'PB',
            'ACVOL_1': 'Volume',
        }

        df = df.rename(columns=column_mapping)

        # Parse underlying from option name
        if 'NAME' in df.columns:
            df['Under'] = df['NAME'].str.split().str[0]

        # Ensure CP column has standard values
        if 'CP' in df.columns:
            df['CP'] = df['CP'].str.upper()
            df['CP'] = df['CP'].map({'C': 'CALL', 'P': 'PUT', 'CALL': 'CALL', 'PUT': 'PUT'})

        return df
    except Exception as e:
        print(f"Error retrieving snapshot: {e}")
        return pd.DataFrame()

print("="*80)
print("RETRIEVING OPTIONS DATA FROM LSEG")
print("="*80)

# Step 1: Get option universe from Datastream
print("\\n1. Retrieving option universe from Datastream...")
meta_list = []
successful_lists = []
failed_lists = []

for option_list in OPTION_LISTS:
    try:
        data = ds.get_data(tickers=option_list, fields=['RIC', 'NAME', 'OXPD'], kind=0)

        if data is not None and len(data) > 0:
            if 'Value' in data.columns and 'Datatype' in data.columns:
                ric_count = len(data[data['Datatype'] == 'RIC'])
                if ric_count > 0:
                    meta_list.append(data)
                    successful_lists.append(option_list)
                    print(f"  ✓ {option_list}: {ric_count} options")
                else:
                    failed_lists.append(option_list)
        else:
            failed_lists.append(option_list)

        sleep(0.1)
    except Exception as e:
        failed_lists.append(option_list)
        print(f"  ✗ {option_list}: {str(e)[:50]}")

print(f"\\nDatastream Summary: {len(successful_lists)}/{len(OPTION_LISTS)} lists retrieved")

if meta_list:
    metadata = pd.concat(meta_list, ignore_index=True)

    # Process metadata
    ric_data = metadata[metadata['Datatype'] == 'RIC'].copy()
    name_data = metadata[metadata['Datatype'] == 'NAME'].copy()
    expiry_data = metadata[metadata['Datatype'] == 'OXPD'].copy()

    options_meta_ds = ric_data.merge(name_data, left_on='Instrument', right_on='Instrument', suffixes=('_ric', '_name'))
    options_meta_ds = options_meta_ds[['Instrument', 'Value_ric', 'Value_name']]
    options_meta_ds.columns = ['Instrument', 'RIC', 'NAME']

    options_meta_ds = options_meta_ds.merge(expiry_data, left_on='Instrument', right_on='Instrument')
    options_meta_ds = options_meta_ds[['Instrument', 'RIC', 'NAME', 'Value']]
    options_meta_ds.columns = ['Instrument', 'RIC', 'NAME', 'EXP']
    options_meta_ds = options_meta_ds[options_meta_ds['RIC'] != 'NA']

    # Parse option names
    try:
        options_meta_ds[['CP', 'Under', '_temp', 'Strike']] = options_meta_ds['NAME'].str.split(' ', expand=True)
        options_meta_ds = options_meta_ds.drop(columns=['_temp'])
    except:
        options_meta_ds['CP'] = options_meta_ds['NAME'].str.split().str[0]
        options_meta_ds['Under'] = options_meta_ds['NAME'].str.split().str[1]
        options_meta_ds['Strike'] = options_meta_ds['NAME'].str.split().str[3]

    all_option_rics = options_meta_ds['RIC'].unique().tolist()
    print(f"✓ Extracted {len(all_option_rics)} unique option RICs")

    available_tickers = options_meta_ds['Under'].unique()
    print(f"✓ Options available for: {', '.join(sorted(available_tickers))}")

else:
    print("✗ No option metadata retrieved")
    all_option_rics = []
    options_meta_ds = pd.DataFrame()

# Step 2: Get live pricing from LSEG
if len(all_option_rics) > 0:
    print(f"\\n2. Retrieving live pricing from LSEG...")
    print(f"   Processing {len(all_option_rics)} RICs in batches of 50...")

    batch_size = 50
    options_data_list = []
    failed_batches = 0

    for i in range(0, len(all_option_rics), batch_size):
        batch_rics = all_option_rics[i:i+batch_size]
        batch_data = get_live_options_quotes(batch_rics)
        sleep(0.5)

        if not batch_data.empty:
            options_data_list.append(batch_data)
        else:
            failed_batches += 1

        if (i // batch_size + 1) % 10 == 0:
            print(f"   Processed {i // batch_size + 1} batches...")

    if options_data_list:
        options_meta = pd.concat(options_data_list, ignore_index=True)
        options_meta = options_meta[options_meta['RIC'].notna()]

        if 'Strike' in options_meta.columns:
            options_meta['Strike'] = pd.to_numeric(options_meta['Strike'], errors='coerce')

        options_meta['CP'].fillna('PUT', inplace=True)
        options_meta['DTE'] = options_meta['EXP'].apply(calculate_dte)

        print(f"\\n✓ Retrieved live data for {len(options_meta)} options")
        print(f"  Failed batches: {failed_batches}")

        # Show summary by ticker
        ticker_counts = options_meta.groupby('Under').size().sort_values(ascending=False)
        print(f"\\nOptions by ticker:")
        for ticker, count in ticker_counts.items():
            print(f"  {ticker}: {count} options")
    else:
        options_meta = pd.DataFrame()
        print("✗ No live pricing data retrieved")
else:
    options_meta = pd.DataFrame()
    print("\\n✗ Skipping live pricing - no RICs available")

# Step 3: Get underlying prices
if len(options_meta) > 0:
    print(f"\\n3. Retrieving underlying prices...")

    try:
        underlying_df = ld.get_data(
            universe=UNDERLYING_RICS,
            fields=['CF_LAST', 'CF_BID', 'CF_ASK', 'CF_CLOSE']
        )

        if not underlying_df.empty:
            underlying_price = underlying_df.rename(columns={
                'Instrument': 'RIC',
                'CF_LAST': 'P',
            })

            underlying_price['Instrument_'] = underlying_price['RIC'].str.replace('.K', '').str.replace('.O', '')
            underlying_price['Dates'] = pd.Timestamp.now().normalize()

            print(f"✓ Retrieved {len(underlying_price)} underlying prices")
            for _, row in underlying_price.iterrows():
                print(f"  {row['Instrument_']}: ${row['P']:.2f}")
        else:
            print("✗ No underlying prices retrieved")
            underlying_price = pd.DataFrame()
    except Exception as e:
        print(f"✗ Error: {e}")
        underlying_price = pd.DataFrame()
else:
    underlying_price = pd.DataFrame()
    print("\\n✗ Skipping underlying prices - no options data")

print(f"\\n{'='*80}")
print("DATA RETRIEVAL COMPLETE")
print(f"{'='*80}")
print(f"Options: {len(options_meta)}")
print(f"Underlyings: {len(underlying_price)}")
print(f"{'='*80}")


RETRIEVING OPTIONS DATA FROM LSEG
\n1. Retrieving option universe from Datastream...
  ✓ LOPTUPROLC: 274 options
  ✓ LOPTUPROLP: 274 options
  ✓ LOPTSPXULC: 402 options
  ✓ LOPTSPXULP: 402 options
  ✓ LOPTTQQQLC: 1 options
  ✓ LOPTTQQQLP: 1 options
  ✓ LOPTSQQQLC: 1 options
  ✓ LOPTSQQQLP: 1 options
  ✓ LOPTSOXLLC: 1 options
  ✓ LOPTSOXLLP: 1 options
  ✓ LOPTSOXSLC: 1 options
  ✓ LOPTSOXSLP: 1 options
  ✓ LOPTTNALC: 248 options
  ✓ LOPTTNALP: 248 options
  ✓ LOPTTZALC: 18 options
  ✓ LOPTTZALP: 18 options
  ✓ LOPTFASLC: 1 options
  ✓ LOPTFASLP: 1 options
  ✓ LOPTFAZLC: 18 options
  ✓ LOPTFAZLP: 18 options
  ✓ LOPTSPYLC: 2858 options
  ✓ LOPTSPYLP: 2858 options
  ✓ LOPTSHLC: 22 options
  ✓ LOPTSHLP: 22 options
  ✓ LOPTQQQLC: 2415 options
  ✓ LOPTQQQLP: 2415 options
  ✓ LOPTPSQLC: 1 options
  ✓ LOPTPSQLP: 1 options
  ✓ LOPTIWMLC: 967 options
  ✓ LOPTIWMLP: 967 options
  ✓ LOPTRWMLC: 21 options
  ✓ LOPTRWMLP: 21 options
  ✓ LOPTDIALC: 1 options
  ✓ LOPTDIALP: 1 options
  ✓ LOPTDOGLC: 86 o

In [5]:
# 5. Process Options Data - Merge with Underlying Prices

if not options_meta.empty and not underlying_price.empty:
    print("Processing options data...")

    # Add current date
    options_meta['Dates'] = pd.Timestamp.now().normalize()

    # Merge with underlying prices
    options_data = options_meta.merge(
        underlying_price[['Instrument_', 'P', 'Dates']],
        left_on=['Under', 'Dates'],
        right_on=['Instrument_', 'Dates'],
        how='left'
    )

    # Rename for compatibility
    options_data = options_data.rename(columns={
        'MP': 'Option_Price',
        'VL': 'IV',
        'P': 'Under_Price',
        'RIC': 'Instrument'
    })

    # Required columns
    required_cols = ['Dates', 'Under', 'Under_Price', 'Strike', 'Instrument',
                     'Option_Price', 'IV', 'PA', 'PB', 'EXP', 'DTE', 'CP']

    available_cols = [col for col in required_cols if col in options_data.columns]
    options_data = options_data[available_cols]

    # Convert to copy to avoid view warnings
    options_data = options_data.copy()

    # Ensure numeric columns are float
    for col in ['Option_Price', 'PA', 'PB', 'Strike', 'Under_Price', 'IV']:
        if col in options_data.columns:
            options_data[col] = pd.to_numeric(options_data[col], errors='coerce').astype('float64')

    # Backfill missing option prices with mid-price
    if 'Option_Price' in options_data.columns:
        mask = options_data['Option_Price'].isna()
        if mask.sum() > 0:
            mid_price = (options_data.loc[mask, 'PA'] + options_data.loc[mask, 'PB']) / 2
            options_data.loc[mask, 'Option_Price'] = mid_price.values
            print(f"  Backfilled {mask.sum()} missing option prices")

    # Filter out options with missing data
    before = len(options_data)
    options_data = options_data[options_data['Under_Price'].notna()]
    options_data = options_data[options_data['Strike'].notna()]
    after = len(options_data)

    if before > after:
        print(f"  Filtered out {before - after} options with missing data")

    # Calculate moneyness
    options_data['moneyness'] = np.log(options_data['Strike'] / options_data['Under_Price'])

    # Add time to expiry
    options_data['time_to_expiry'] = options_data['DTE'] / 365.0

    print(f"✓ Processed {len(options_data)} options with complete data")

    # Show summary
    ticker_summary = options_data.groupby(['Under', 'CP']).size().unstack(fill_value=0)
    print(f"\\nOptions summary:")
    print(ticker_summary)

    display(options_data.head())

else:
    print("✗ Cannot process - missing options or underlying data")
    options_data = pd.DataFrame()


Processing options data...
  Backfilled 15650 missing option prices
  Filtered out 9750 options with missing data
✓ Processed 5900 options with complete data
\nOptions summary:
CP     CALL   PUT
Under            
QQQ    2415  2415
SPXU    262   262
UPRO    272   274


,Dates,Under,Under_Price,Strike,Instrument,Option_Price,IV,PA,PB,EXP,DTE,CP,moneyness,time_to_expiry
0,2026-02-18,UPRO,115.15,70.0,UPROB202607000.U,45.10,294.3174,47.0,43.2,2026-02-20,1.0,CALL,-0.497740,0.00274
1,2026-02-18,UPRO,115.15,75.0,UPROB202607500.U,40.10,262.2680,42.0,38.2,2026-02-20,1.0,CALL,-0.428748,0.00274
2,2026-02-18,UPRO,115.15,80.0,UPROB202608000.U,35.20,227.8655,37.0,33.4,2026-02-20,1.0,CALL,-0.364209,0.00274
3,2026-02-18,UPRO,115.15,85.0,UPROB202608500.U,30.65,163.0516,32.0,29.3,2026-02-20,1.0,CALL,-0.303584,0.00274
4,2026-02-18,UPRO,115.15,90.0,UPROB202609000.U,25.60,161.0571,27.0,24.2,2026-02-20,1.0,CALL,-0.246426,0.00274


In [6]:
# 6. Identify Arbitrage Opportunities Across ETF Pairs

all_opportunities_bull_call = []
all_opportunities_bear_call = []

if len(options_data) > 0:
    print("\\n" + "="*80)
    print("ANALYZING ETF PAIRS FOR ARBITRAGE OPPORTUNITIES (DELTA-NEUTRAL)")
    print("="*80)

    for pair in ETF_PAIRS:
        bull_ticker = pair["bull"]
        bear_ticker = pair["bear"]
        pair_name_simple = f"{bull_ticker}/{bear_ticker}"
        underlying_vol = pair["underlying_vol"]
        leverage = pair.get("leverage", 1.0)  # Get pair-specific leverage

        print(f"\n--- {pair_name_simple} (σ={underlying_vol:.0%}, {leverage}x) ---")

        # Get options for each ticker
        bull_opts = options_data[options_data['Under'] == bull_ticker].copy()
        bear_opts = options_data[options_data['Under'] == bear_ticker].copy()

        if len(bull_opts) == 0 or len(bear_opts) == 0:
            print(f"  ⚠ Insufficient data ({len(bull_opts)} bull, {len(bear_opts)} bear)")
            continue

        print(f"  Options: {len(bull_opts)} {bull_ticker}, {len(bear_opts)} {bear_ticker}")

        # Separate calls and puts
        bull_calls = bull_opts[bull_opts['CP'] == 'CALL'].copy()
        bull_puts = bull_opts[bull_opts['CP'] == 'PUT'].copy()
        bear_calls = bear_opts[bear_opts['CP'] == 'CALL'].copy()
        bear_puts = bear_opts[bear_opts['CP'] == 'PUT'].copy()

        # Strategy 1: Bull Call undervalued - Long Bull Call / Short Bear Put
        if len(bull_calls) > 0 and len(bear_puts) > 0:
            for exp_date in bull_calls['EXP'].unique():
                bull_exp = bull_calls[bull_calls['EXP'] == exp_date]
                bear_exp = bear_puts[bear_puts['EXP'] == exp_date]

                if len(bull_exp) == 0 or len(bear_exp) == 0:
                    continue

                for _, bull_row in bull_exp.iterrows():
                    # Filter DTE range
                    if bull_row['DTE'] < MIN_DTE or bull_row['DTE'] > MAX_DTE:
                        continue

                    # Calculate erosion-adjusted target moneyness
                    T = bull_row['DTE'] / 365.0
                    bull_moneyness = bull_row['moneyness']
                    target_bear_moneyness = -bull_moneyness - EROSION * T

                    # Filter bear puts by adjusted moneyness
                    bear_candidates = bear_exp[
                        np.abs(bear_exp['moneyness'] - target_bear_moneyness) < 0.15
                    ].to_dict('records')

                    if len(bear_candidates) == 0:
                        continue

                    # Find delta-neutral pairing
                    best_pairing = find_delta_neutral_pair(
                        bull_row.to_dict(),
                        bear_candidates,
                        max_total_contracts=12
                    )

                    if best_pairing is None:
                        continue

                    bear_row = best_pairing['short_option']

                    # Calculate IV gap
                    if pd.notna(bull_row['IV']) and pd.notna(bear_row['IV']):
                        iv_gap = bull_row['IV'] - bear_row['IV']

                        # Look for negative IV gap (bull call undervalued)
                        if iv_gap < -2 and pd.notna(bull_row['PA']) and pd.notna(bear_row['PB']):
                            # Generate standardized pair name: undervalued first
                            pair_name_std = standardize_pair_name(
                                bull_ticker, 'C', bull_row['Strike'],
                                bear_ticker, 'P', bear_row['Strike'],
                                exp_date
                            )

                            opp = {
                                'pair_name': pair_name_std,
                                'pair_simple': pair_name_simple,
                                'strategy': 'Buy Bull Call / Sell Bear Put',
                                'bull_ticker': bull_ticker,
                                'bear_ticker': bear_ticker,
                                'bull_option_type': 'CALL',
                                'bear_option_type': 'PUT',
                                'bull_strike': bull_row['Strike'],
                                'bear_strike': bear_row['Strike'],
                                'num_bull': best_pairing['num_long'],
                                'num_bear': best_pairing['num_short'],
                                'total_contracts': best_pairing['total_contracts'],
                                'delta_sum': best_pairing['delta_sum'],
                                'delta_neutrality': best_pairing['delta_neutrality_score'],
                                'bull_price_ask': bull_row['PA'],
                                'bear_price_bid': bear_row['PB'],
                                'bull_iv': bull_row['IV'],
                                'bear_iv': bear_row['IV'],
                                'iv_gap': iv_gap,
                                'expiry': exp_date,
                                'dte': int(bull_row['DTE']),
                                'moneyness': bull_row['moneyness'],
                                'moneyness_bull': bull_row['moneyness'],
                                'moneyness_bear': bear_row['moneyness'],
                                'bull_under_price': bull_row['Under_Price'],
                                'bear_under_price': bear_row['Under_Price'],
                                'underlying_vol': underlying_vol,
                                'leverage': leverage,
                            }
                            all_opportunities_bull_call.append(opp)

        # Strategy 2: Bear Call undervalued - Long Bear Call / Short Bull Put
        if len(bear_calls) > 0 and len(bull_puts) > 0:
            for exp_date in bear_calls['EXP'].unique():
                bear_exp = bear_calls[bear_calls['EXP'] == exp_date]
                bull_exp = bull_puts[bull_puts['EXP'] == exp_date]

                if len(bear_exp) == 0 or len(bull_exp) == 0:
                    continue

                for _, bear_row in bear_exp.iterrows():
                    if bear_row['DTE'] < MIN_DTE or bear_row['DTE'] > MAX_DTE:
                        continue

                    # Calculate erosion-adjusted target moneyness
                    T = bear_row['DTE'] / 365.0
                    bear_moneyness = bear_row['moneyness']
                    target_bull_moneyness = -bear_moneyness - EROSION * T

                    # Filter bull puts by adjusted moneyness
                    bull_candidates = bull_exp[
                        np.abs(bull_exp['moneyness'] - target_bull_moneyness) < 0.15
                    ].to_dict('records')

                    if len(bull_candidates) == 0:
                        continue

                    # Find delta-neutral pairing
                    best_pairing = find_delta_neutral_pair(
                        bear_row.to_dict(),
                        bull_candidates,
                        max_total_contracts=12
                    )

                    if best_pairing is None:
                        continue

                    bull_row = best_pairing['short_option']

                    if pd.notna(bear_row['IV']) and pd.notna(bull_row['IV']):
                        iv_gap = bear_row['IV'] - bull_row['IV']

                        if iv_gap < -2 and pd.notna(bear_row['PA']) and pd.notna(bull_row['PB']):
                            # Generate standardized pair name: undervalued first (bear in this case)
                            pair_name_std = standardize_pair_name(
                                bear_ticker, 'C', bear_row['Strike'],
                                bull_ticker, 'P', bull_row['Strike'],
                                exp_date
                            )

                            opp = {
                                'pair_name': pair_name_std,
                                'pair_simple': pair_name_simple,
                                'strategy': 'Buy Bear Call / Sell Bull Put',
                                'bull_ticker': bull_ticker,
                                'bear_ticker': bear_ticker,
                                'bull_option_type': 'PUT',
                                'bear_option_type': 'CALL',
                                'bull_strike': bull_row['Strike'],
                                'bear_strike': bear_row['Strike'],
                                'num_bear': best_pairing['num_long'],
                                'num_bull': best_pairing['num_short'],
                                'total_contracts': best_pairing['total_contracts'],
                                'delta_sum': best_pairing['delta_sum'],
                                'delta_neutrality': best_pairing['delta_neutrality_score'],
                                'bull_price_bid': bull_row['PB'],
                                'bear_price_ask': bear_row['PA'],
                                'bull_iv': bull_row['IV'],
                                'bear_iv': bear_row['IV'],
                                'iv_gap': iv_gap,
                                'expiry': exp_date,
                                'dte': int(bear_row['DTE']),
                                'moneyness': bear_row['moneyness'],
                                'moneyness_bull': bull_row['moneyness'],
                                'moneyness_bear': bear_row['moneyness'],
                                'bull_under_price': bull_row['Under_Price'],
                                'bear_under_price': bear_row['Under_Price'],
                                'underlying_vol': underlying_vol,
                                'leverage': leverage,
                            }
                            all_opportunities_bear_call.append(opp)

        bull_count = len([o for o in all_opportunities_bull_call if o['pair_simple'] == pair_name_simple])
        bear_count = len([o for o in all_opportunities_bear_call if o['pair_simple'] == pair_name_simple])
        print(f"  Opportunities: {bull_count} bull call, {bear_count} bear call")

    print(f"\\n{'='*80}")
    print(f"TOTAL OPPORTUNITIES FOUND")
    print(f"{'='*80}")
    print(f"  Bull Call Strategy: {len(all_opportunities_bull_call)}")
    print(f"  Bear Call Strategy: {len(all_opportunities_bear_call)}")
    print(f"  Total: {len(all_opportunities_bull_call) + len(all_opportunities_bear_call)}")

else:
    print("✗ No data available for opportunity analysis")


\n================================================================================
ANALYZING ETF PAIRS FOR ARBITRAGE OPPORTUNITIES (DELTA-NEUTRAL)

--- UPRO/SPXU (σ=15%, 3.0x) ---
  Options: 546 UPRO, 524 SPXU
  Opportunities: 143 bull call, 44 bear call

--- TQQQ/SQQQ (σ=22%, 3.0x) ---
  ⚠ Insufficient data (0 bull, 0 bear)

--- SOXL/SOXS (σ=30%, 3.0x) ---
  ⚠ Insufficient data (0 bull, 0 bear)

--- TNA/TZA (σ=18%, 3.0x) ---
  ⚠ Insufficient data (0 bull, 0 bear)

--- FAS/FAZ (σ=25%, 3.0x) ---
  ⚠ Insufficient data (0 bull, 0 bear)

--- SPY/SH (σ=15%, 1.0x) ---
  ⚠ Insufficient data (0 bull, 0 bear)

--- QQQ/PSQ (σ=22%, 1.0x) ---
  ⚠ Insufficient data (4830 bull, 0 bear)

--- IWM/RWM (σ=18%, 1.0x) ---
  ⚠ Insufficient data (0 bull, 0 bear)

--- DIA/DOG (σ=14%, 1.0x) ---
  ⚠ Insufficient data (0 bull, 0 bear)

--- XLF/SEF (σ=25%, 1.0x) ---
  ⚠ Insufficient data (0 bull, 0 bear)

--- XLE/ERY (σ=35%, 1.0x) ---
  ⚠ Insufficient data (0 bull, 0 bear)

--- NVDA/SOXS (σ=45%, 1.0x) ---
  ⚠ In

In [7]:
# 7. Calculate P&L Projections with Volatility Decay Model

print("\\n" + "="*80)
print("CALCULATING P&L PROJECTIONS WITH DECAY MODEL")
print("="*80)

# Calculate P&L for all opportunities using the volatility decay model
if len(all_opportunities_bull_call) > 0:
    print("\\nCalculating P&L projections for bull call opportunities...")

    for opp in all_opportunities_bull_call:
        try:
            T = opp['dte'] / 365.0
            leverage = opp.get('leverage', 3.0)  # Get leverage from opportunity

            # Use helper function with decay model
            pnl_results = calculate_pnl_with_decay(
                strategy_type='bull_call',
                bull_strike=opp['bull_strike'],
                bear_strike=opp['bear_strike'],
                bull_price=opp['bull_price_ask'],
                bear_price=opp['bear_price_bid'],
                long_price=opp['bull_price_ask'],
                short_price=opp['bear_price_bid'],
                bull_underlying=opp['bull_under_price'],
                bear_underlying=opp['bear_under_price'],
                sigma_underlying=opp['underlying_vol'],
                T=T,
                leverage=leverage
            )

            # Update opportunity with P&L results
            opp.update(pnl_results)

            # Calculate breakeven
            opp['breakeven'] = calculate_breakeven_price(
                'bull_call',
                opp['bull_strike'],
                opp['bear_strike'],
                pnl_results['net_cost']
            )

        except Exception as e:
            print(f"  Error calculating P&L for {opp['pair_name']}: {e}")
            continue

    # Filter out opportunities without P&L
    all_opportunities_bull_call = [o for o in all_opportunities_bull_call if 'pnl_expected' in o]
    print(f"  ✓ Calculated P&L for {len(all_opportunities_bull_call)} opportunities")

if len(all_opportunities_bear_call) > 0:
    print("\\nCalculating P&L projections for bear call opportunities...")

    for opp in all_opportunities_bear_call:
        try:
            T = opp['dte'] / 365.0
            leverage = opp.get('leverage', 3.0)  # Get leverage from opportunity

            # Use helper function with decay model
            pnl_results = calculate_pnl_with_decay(
                strategy_type='bear_call',
                bull_strike=opp['bull_strike'],
                bear_strike=opp['bear_strike'],
                bull_price=opp['bull_price_bid'],
                bear_price=opp['bear_price_ask'],
                long_price=opp['bear_price_ask'],
                short_price=opp['bull_price_bid'],
                bull_underlying=opp['bull_under_price'],
                bear_underlying=opp['bear_under_price'],
                sigma_underlying=opp['underlying_vol'],
                T=T,
                leverage=leverage
            )

            # Update opportunity with P&L results
            opp.update(pnl_results)

            # Calculate breakeven
            opp['breakeven'] = calculate_breakeven_price(
                'bear_call',
                opp['bull_strike'],
                opp['bear_strike'],
                pnl_results['net_cost']
            )

        except Exception as e:
            print(f"  Error calculating P&L for {opp['pair_name']}: {e}")
            continue

    # Filter out opportunities without P&L
    all_opportunities_bear_call = [o for o in all_opportunities_bear_call if 'pnl_expected' in o]
    print(f"  ✓ Calculated P&L for {len(all_opportunities_bear_call)} opportunities")

print(f"\\n{'='*80}")
print("P&L CALCULATION COMPLETE")
print(f"{'='*80}")


\n================================================================================
CALCULATING P&L PROJECTIONS WITH DECAY MODEL
\nCalculating P&L projections for bull call opportunities...
  ✓ Calculated P&L for 143 opportunities
\nCalculating P&L projections for bear call opportunities...
  ✓ Calculated P&L for 44 opportunities
\n================================================================================
P&L CALCULATION COMPLETE


In [8]:
# 8. Display Top Opportunities with P&L Projections

print("\\n" + "="*80)
print("TOP OPPORTUNITIES WITH P&L PROJECTIONS")
print("="*80)

top_n = 20

if len(all_opportunities_bull_call) > 0:
    # Convert to DataFrame and sort
    df_bull_call_opps = pd.DataFrame(all_opportunities_bull_call)
    df_bull_call_opps = df_bull_call_opps.sort_values('pnl_expected', ascending=False).head(top_n)

    print(f"\\n{'='*80}")
    print(f"LIST 1: TOP {len(df_bull_call_opps)} BULL CALL OPPORTUNITIES")
    print(f"{'='*80}")
    print(f"Strategy: Long Bull Call + Short Bear Put")
    print(f"Rationale: Profit when bull ETF rises, hedged by shorting bear ETF puts\\n")

    # Display key columns
    display_cols = ['pair_name', 'bull_strike', 'bear_strike', 'iv_gap', 'dte',
                   'net_cost', 'pnl_expected', 'pnl_best', 'pnl_worst']

    df_display = df_bull_call_opps[display_cols].copy()
    df_display.columns = ['Pair', 'Bull Strike', 'Bear Strike', 'IV Gap (%)', 'DTE',
                          'Net Cost ($)', 'Expected P&L ($)', 'Best Case ($)', 'Worst Case ($)']

    display(df_display)

    # Summary statistics
    print(f"\\nSummary Statistics:")
    print(f"  Avg Expected P&L: ${df_bull_call_opps['pnl_expected'].mean():.2f}")
    print(f"  Avg IV Gap: {df_bull_call_opps['iv_gap'].mean():.2f}%")
    print(f"  Avg DTE: {df_bull_call_opps['dte'].mean():.0f} days")
    print(f"  Best Opportunity: {df_bull_call_opps.iloc[0]['pair_name']} - Expected P&L: ${df_bull_call_opps.iloc[0]['pnl_expected']:.2f}")

    # Pairs with most opportunities
    pairs_with_opps = df_bull_call_opps.groupby('pair_name').size().sort_values(ascending=False)
    print(f"\\n  Top Pairs:")
    for pair, count in pairs_with_opps.head(5).items():
        print(f"    {pair}: {count} opportunities")

    # Export to CSV
    csv_path_bull = os.path.join(HTML_OUTPUT_DIR, f'bull_call_opportunities_{TIMESTAMP}.csv')
    df_bull_call_opps.to_csv(csv_path_bull, index=False)
    print(f"\\n✓ Exported to: {csv_path_bull}")

else:
    print("\\n✗ No bull call opportunities found")
    df_bull_call_opps = pd.DataFrame()

if len(all_opportunities_bear_call) > 0:
    # Convert to DataFrame and sort
    df_bear_call_opps = pd.DataFrame(all_opportunities_bear_call)
    df_bear_call_opps = df_bear_call_opps.sort_values('pnl_expected', ascending=False).head(top_n)

    print(f"\\n{'='*80}")
    print(f"LIST 2: TOP {len(df_bear_call_opps)} BEAR CALL OPPORTUNITIES")
    print(f"{'='*80}")
    print(f"Strategy: Long Bear Call + Short Bull Put")
    print(f"Rationale: Profit when underlying falls, hedged by shorting bull ETF puts\\n")

    display_cols = ['pair_name', 'bull_strike', 'bear_strike', 'iv_gap', 'dte',
                   'net_cost', 'pnl_expected', 'pnl_best', 'pnl_worst']

    df_display = df_bear_call_opps[display_cols].copy()
    df_display.columns = ['Pair', 'Bull Strike', 'Bear Strike', 'IV Gap (%)', 'DTE',
                          'Net Cost ($)', 'Expected P&L ($)', 'Best Case ($)', 'Worst Case ($)']

    display(df_display)

    print(f"\\nSummary Statistics:")
    print(f"  Avg Expected P&L: ${df_bear_call_opps['pnl_expected'].mean():.2f}")
    print(f"  Avg IV Gap: {df_bear_call_opps['iv_gap'].mean():.2f}%")
    print(f"  Avg DTE: {df_bear_call_opps['dte'].mean():.0f} days")
    print(f"  Best Opportunity: {df_bear_call_opps.iloc[0]['pair_name']} - Expected P&L: ${df_bear_call_opps.iloc[0]['pnl_expected']:.2f}")

    pairs_with_opps = df_bear_call_opps.groupby('pair_name').size().sort_values(ascending=False)
    print(f"\\n  Top Pairs:")
    for pair, count in pairs_with_opps.head(5).items():
        print(f"    {pair}: {count} opportunities")

    # Export to CSV
    csv_path_bear = os.path.join(HTML_OUTPUT_DIR, f'bear_call_opportunities_{TIMESTAMP}.csv')
    df_bear_call_opps.to_csv(csv_path_bear, index=False)
    print(f"\\n✓ Exported to: {csv_path_bear}")

else:
    print("\\n✗ No bear call opportunities found")
    df_bear_call_opps = pd.DataFrame()

print(f"\\n{'='*80}")
print("ANALYSIS COMPLETE")
print(f"{'='*80}")


\n================================================================================
TOP OPPORTUNITIES WITH P&L PROJECTIONS
\n================================================================================
LIST 1: TOP 20 BULL CALL OPPORTUNITIES
Strategy: Long Bull Call + Short Bear Put
Rationale: Profit when bull ETF rises, hedged by shorting bear ETF puts\n


,Pair,Bull Strike,Bear Strike,IV Gap (%),DTE,Net Cost ($),Expected P&L ($),Best Case ($),Worst Case ($)
6,UPRO_C_86_SPXU_P_57_20260320,86.0,57.0,-7.5082,29,23.50,0.720541,10.534487,-17.749786
3,UPRO_C_83_SPXU_P_62_20260320,83.0,62.0,-22.3058,29,21.80,0.420541,10.234487,-15.936036
8,UPRO_C_88_SPXU_P_59_20260320,88.0,59.0,-7.2483,29,19.90,0.320541,10.134487,-16.149786
4,UPRO_C_84_SPXU_P_63_20260320,84.0,63.0,-12.1517,29,20.00,0.220541,10.034487,-16.136036
0,UPRO_C_75_SPXU_P_75_20260320,75.0,75.0,-9.9328,29,17.00,0.220541,10.034487,-16.136036
7,UPRO_C_87_SPXU_P_58_20260320,87.0,58.0,-6.6685,29,22.00,0.220541,10.034487,-17.249786
5,UPRO_C_85_SPXU_P_64_20260320,85.0,64.0,-6.6699,29,18.10,0.120541,9.934487,-16.236036
9,UPRO_C_89_SPXU_P_59_20260320,89.0,59.0,-6.6496,29,19.10,0.120541,9.934487,-16.349786
70,UPRO_C_150_SPXU_P_43_20260618,150.0,43.0,-16.8753,119,-0.05,0.050000,-4.029715,0.050000
10,UPRO_C_90_SPXU_P_60_20260320,90.0,60.0,-10.0538,29,17.20,0.020541,9.834487,-16.336036


\nSummary Statistics:
  Avg Expected P&L: $0.00
  Avg IV Gap: -12.86%
  Avg DTE: 38 days
  Best Opportunity: UPRO_C_86_SPXU_P_57_20260320 - Expected P&L: $0.72
\n  Top Pairs:
    UPRO_C_140_SPXU_P_40_20260320: 1 opportunities
    UPRO_C_145_SPXU_P_41_20260320: 1 opportunities
    UPRO_C_150_SPXU_P_43_20260618: 1 opportunities
    UPRO_C_180_SPXU_P_30_20260618: 1 opportunities
    UPRO_C_75_SPXU_P_75_20260320: 1 opportunities
\n✓ Exported to: c:\Users\P3004204\OneDrive - Corporate\Workspace\python\LSEG_Options_Analysis\html_reports\bull_call_opportunities_20260218_211112.csv
\n================================================================================
LIST 2: TOP 20 BEAR CALL OPPORTUNITIES
Strategy: Long Bear Call + Short Bull Put
Rationale: Profit when underlying falls, hedged by shorting bull ETF puts\n


,Pair,Bull Strike,Bear Strike,IV Gap (%),DTE,Net Cost ($),Expected P&L ($),Best Case ($),Worst Case ($)
26,SPXU_C_44_UPRO_P_110_20260618,110.0,44.0,-12.8476,119,-0.80,5.484654,-4.207547,0.80
33,SPXU_C_53_UPRO_P_106_20260618,106.0,53.0,-2.7409,119,-2.50,2.500000,-7.507547,2.50
32,SPXU_C_52_UPRO_P_104_20260618,104.0,52.0,-6.2413,119,-1.70,1.700000,-5.307547,1.70
39,SPXU_C_40_UPRO_P_120_20260918,120.0,40.0,-2.6324,211,-5.00,1.672659,-7.896653,5.00
41,SPXU_C_50_UPRO_P_100_20260918,100.0,50.0,-2.0730,211,-1.20,1.635183,-1.696653,1.20
29,SPXU_C_49_UPRO_P_99_20260618,99.0,49.0,-12.3006,119,0.50,1.582874,0.492453,-0.50
27,SPXU_C_46_UPRO_P_115_20260618,115.0,46.0,-6.6602,119,-3.40,1.084654,-8.607547,3.40
31,SPXU_C_51_UPRO_P_102_20260618,102.0,51.0,-7.3511,119,-0.90,0.982874,-3.107547,0.90
6,SPXU_C_49_UPRO_P_100_20260320,100.0,49.0,-19.6788,29,1.75,0.974531,0.113964,-1.75
11,SPXU_C_54_UPRO_P_106_20260320,106.0,54.0,-5.0857,29,-0.90,0.900000,-8.236036,0.90


\nSummary Statistics:
  Avg Expected P&L: $1.07
  Avg IV Gap: -8.24%
  Avg DTE: 83 days
  Best Opportunity: SPXU_C_44_UPRO_P_110_20260618 - Expected P&L: $5.48
\n  Top Pairs:
    SPXU_C_40_UPRO_P_120_20260918: 1 opportunities
    SPXU_C_44_UPRO_P_110_20260618: 1 opportunities
    SPXU_C_46_UPRO_P_115_20260320: 1 opportunities
    SPXU_C_46_UPRO_P_115_20260618: 1 opportunities
    SPXU_C_49_UPRO_P_100_20260320: 1 opportunities
\n✓ Exported to: c:\Users\P3004204\OneDrive - Corporate\Workspace\python\LSEG_Options_Analysis\html_reports\bear_call_opportunities_20260218_211112.csv
\n================================================================================
ANALYSIS COMPLETE


In [9]:
# 9. Advanced Visualizations and Interactive HTML Export

import plotly.graph_objects as go

print("\\n" + "="*80)
print("CREATING VISUALIZATIONS")
print("="*80)

# Prepare DataFrames
df_bull_call_opps = pd.DataFrame(all_opportunities_bull_call)
df_bear_call_opps = pd.DataFrame(all_opportunities_bear_call)

if len(df_bull_call_opps) > 0 or len(df_bear_call_opps) > 0:

    # 1. Create 3D Surface Plot (IV Gap vs Moneyness and DTE)
    print("\\n1. Creating 3D surface plot...")
    df_all_opps = pd.concat([df_bull_call_opps, df_bear_call_opps], ignore_index=True)
    fig_3d = create_3d_surface_plot(df_all_opps)
    print("   ✓ 3D surface plot created")

    # 2. Create P&L Profile Plots
    print("\\n2. Creating P&L profile plots...")
    if len(df_bull_call_opps) > 0:
        fig_pnl_bull = create_pnl_profile_plots(df_bull_call_opps, 'bull_call', top_n=5)
        print("   ✓ Bull call P&L profiles created")
    else:
        fig_pnl_bull = go.Figure()
        fig_pnl_bull.add_annotation(text="No bull call opportunities",
                                    xref="paper", yref="paper",
                                    x=0.5, y=0.5, showarrow=False)

    if len(df_bear_call_opps) > 0:
        fig_pnl_bear = create_pnl_profile_plots(df_bear_call_opps, 'bear_call', top_n=5)
        print("   ✓ Bear call P&L profiles created")
    else:
        fig_pnl_bear = go.Figure()
        fig_pnl_bear.add_annotation(text="No bear call opportunities",
                                    xref="paper", yref="paper",
                                    x=0.5, y=0.5, showarrow=False)

    # 3. Create Summary Dashboard
    print("\\n3. Creating summary dashboard...")
    fig_summary = create_opportunity_summary_plot(df_bull_call_opps, df_bear_call_opps)
    print("   ✓ Summary dashboard created")

    # 4. Export Interactive HTML Report
    print("\\n4. Exporting interactive HTML report...")
    html_path = os.path.join(HTML_OUTPUT_DIR, f'leveraged_etf_analysis_{TIMESTAMP}.html')

    # Prepare data for HTML export
    display_cols = ['pair_name', 'bull_strike', 'bear_strike', 'iv_gap', 'dte',
                   'net_cost', 'pnl_expected', 'pnl_best', 'pnl_worst',
                   'decay_factor', 'erosion_rate', 'breakeven']

    df_bull_export = df_bull_call_opps[display_cols].copy() if len(df_bull_call_opps) > 0 else pd.DataFrame()
    df_bear_export = df_bear_call_opps[display_cols].copy() if len(df_bear_call_opps) > 0 else pd.DataFrame()

    export_interactive_html(
        df_bull_export, df_bear_export,
        fig_3d, fig_pnl_bull, fig_pnl_bear, fig_summary,
        html_path, TIMESTAMP
    )

    print(f"\\n{'='*80}")
    print("VISUALIZATION COMPLETE")
    print(f"{'='*80}")
    print(f"\\n✓ Interactive HTML report saved to:")
    print(f"  {html_path}")

    # Display figures inline
    print("\\n\\nDisplaying visualizations inline...")

    print("\\n--- 3D Surface Plot ---")
    fig_3d.show()

    print("\\n--- Bull Call P&L Profiles ---")
    fig_pnl_bull.show()

    print("\\n--- Bear Call P&L Profiles ---")
    fig_pnl_bear.show()

    print("\\n--- Summary Dashboard ---")
    fig_summary.show()

else:
    print("\\n⚠ No opportunities found - skipping visualizations")


\n================================================================================
CREATING VISUALIZATIONS
\n1. Creating 3D surface plot...
   ✓ 3D surface plot created
\n2. Creating P&L profile plots...
   ✓ Bull call P&L profiles created
   ✓ Bear call P&L profiles created
\n3. Creating summary dashboard...
   ✓ Summary dashboard created
\n4. Exporting interactive HTML report...
✓ Interactive HTML report exported to: c:\Users\P3004204\OneDrive - Corporate\Workspace\python\LSEG_Options_Analysis\html_reports\leveraged_etf_analysis_20260218_211112.html
\n================================================================================
VISUALIZATION COMPLETE
\n✓ Interactive HTML report saved to:
  c:\Users\P3004204\OneDrive - Corporate\Workspace\python\LSEG_Options_Analysis\html_reports\leveraged_etf_analysis_20260218_211112.html
\n\nDisplaying visualizations inline...
\n--- 3D Surface Plot ---


\n--- Bull Call P&L Profiles ---


\n--- Bear Call P&L Profiles ---


\n--- Summary Dashboard ---


In [10]:
# 10. Cleanup
try:
    ld.close_session()
    print("✓ LSEG session closed")
except:
    pass

print("\n" + "="*80)
print("FINAL SUMMARY")
print("="*80)
print(f"Bull Call Opportunities: {len(df_bull_call_opps) if len(df_bull_call_opps) > 0 else 0}")
print(f"Bear Call Opportunities: {len(df_bear_call_opps) if len(df_bear_call_opps) > 0 else 0}")
print(f"Total Opportunities: {(len(df_bull_call_opps) if len(df_bull_call_opps) > 0 else 0) + (len(df_bear_call_opps) if len(df_bear_call_opps) > 0 else 0)}")
print(f"\nReports saved to: {HTML_OUTPUT_DIR}")
print("="*80)


✓ LSEG session closed

FINAL SUMMARY
Bull Call Opportunities: 143
Bear Call Opportunities: 44
Total Opportunities: 187

Reports saved to: c:\Users\P3004204\OneDrive - Corporate\Workspace\python\LSEG_Options_Analysis\html_reports
